In [3]:
import numpy as np
import pandas as pd

In [43]:
data = pd.read_csv("./input/country_results.csv")

Os dados com que estamos trabalhando referem-se aos resultados de cada país participante da Olimpíada Internacional de Matemática (IMO) ao longo dos anos de 1959 - 2024. Cada linha da tabela abaixo traz informações sobre a participação de um país em uma edição específica da competição. Cada coluna representa um possível dado nesse contexto, sendo eles: 
- ano da edição (year): variável numérica discreta;
- país participante (country): variável categórica nominal;
- quantidade de participantes da equipe (team_size_all): variável numérica discreta;
- quantidade de participantes masculinos da equipe (team_size_male): variável numérica discreta;
- quantidade de participantes femininos da equipe (team_size_female): variável numérica discreta;
- pontuação acumulada da equipe em cada problema respectivo da prova (p1, p2, ..., p7): variável numérica discreta;
- quantidade de medalhas de ouro, prata ou bronze e quantidade de menções honrosas conferidas à equipe (awards_gold, awards_silver, awards_bronze, awards_honorable_mentions): variável numérica discreta;
- nome completo do líder da equipe (leader): variável categórica nominal;
- nome completo do vice-líder da equipe (deputy_leader): variável categórica nominal.


Valores com informações faltantes são marcadas com "NA".

In [236]:
display(data)

,year,country,team_size_all,team_size_male,team_size_female,p1,p2,p3,p4,p5,p6,p7,awards_gold,awards_silver,awards_bronze,awards_honorable_mentions,leader,deputy_leader
0,2024,United States of America,6,5.0,1.0,42.0,41.0,19.0,40.0,35.0,15.0,NaN,5.0,1.0,0.0,0.0,John Berman,Carl Schildkraut
1,2024,People's Republic of China,6,6.0,0.0,42.0,42.0,31.0,40.0,22.0,13.0,NaN,5.0,1.0,0.0,0.0,Liang Xiao,Yijun Yao
2,2024,Republic of Korea,6,6.0,0.0,42.0,37.0,18.0,42.0,7.0,22.0,NaN,2.0,4.0,0.0,0.0,Suyoung Choi,Hwajong Yoo
3,2024,India,6,6.0,0.0,42.0,34.0,11.0,42.0,28.0,10.0,NaN,4.0,1.0,0.0,1.0,Krishnan Sivasubramanian,Rijul Saini
4,2024,Belarus,6,6.0,0.0,42.0,30.0,10.0,42.0,36.0,5.0,NaN,4.0,0.0,2.0,0.0,David Zmiaikou,Dzmitry Bazyleu
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3775,1959,Czechoslovakia,8,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0.0,NaN,Rudolf Zelinka,NaN
3776,1959,Bulgaria,8,6.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,Stoian Budurov,NaN
3777,1959,Poland,8,7.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,Mieczyslaw Czyżykowski,NaN
3778,1959,Union of Soviet Socialist Republics,4,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.0,NaN,Anatol Mikhailovitch Vladimirski,NaN


A fim de possuirmos variáveis de todos os 4 tipos possíveis para esse estudo, iremos derivar duas novas variáveis a partir desses dados:
- proporção de participantes femininos na equipe (female_proportion): variável numérica contínua;
- maturidade da equipe em concentrar-se somente nas fáceis ou garantir pontos nas difíceis (team_approach): variável categórica ordinal.

In [237]:
data["female_rate"] = data["team_size_female"]/data["team_size_all"]

max = 7*data["team_size_all"]*2
p_easy = (data["p1"]+data["p4"])/max
p_medium = (data["p2"]+data["p5"])/max
p_hard = (data["p3"]+data["p6"])/max

thresholds = [
    (p_easy >= 0.8) & (p_medium >= 0.5) & (p_hard >= 0.2),
    (p_easy >= 0.8) & (p_medium >= 0.5),
    (p_easy >= 0.8),
]

values = [3, 2, 1]

data["team_approach"] = np.select(thresholds, values, default=0)

In [ ]:
with pd.option_context("display.max_rows", None):
    display(data[["year", "country", "team_approach"]])

# 1.4 medidas de posicao central

## 1.4.1 - Distribuição geral das medalhas

A partir dos dados gerais observamos algumas tendencias, a media de medalhas é inferior a uma medalha *ouro(0,47)* e *prata(0,96)* sendo *bronze(1,42).* As medianas e modas de ouro e prata convergem para 0, indicando que pela mediana pelo menos metade dos registros validos nao ganhou medalhas e pela moda a maior parter zerou na categoria. Posteriomente, investigaremos valores que podem ter influenciado as nossa medidas.

In [21]:
medalhas = ["awards_gold", "awards_silver", "awards_bronze"]
medalhas_resumo = data[medalhas].agg(["count", "mean", "median",'std'])
display(medalhas_resumo)

,awards_gold,awards_silver,awards_bronze
count,3778.000000,3778.000000,3778.000000
mean,0.470619,0.960296,1.417152
median,0.000000,0.000000,1.000000
std,1.045590,1.285347,1.365307


In [240]:
data[medalhas].mode()

,awards_gold,awards_silver,awards_bronze
0,0.0,0.0,0.0


A análise da frequência de medalhas evidencia uma concentração de registros com valor zero, principalmente nas categorias ouro e prata. Aproximadamente 75,5% dos registros válidos não apresentam medalhas de ouro, enquanto 53,7% não apresentam medalhas de prata. Na categoria bronze, essa proporção é de aproximadamente 33,7%. 
---
Essa densidade de valores nulos contribui para explicar os resultados anteriores, a mediana do ouro e prata, por exemplo, indicam valores iguais a zero. Além disso, a discrepancia entre os valores maximos e minimos para alem do que ja foi discuido pode indicar uma assimetria positiva.

In [241]:
filtro = data["awards_gold"].isin([0.0,6.0])
data[filtro]['awards_gold'].value_counts()

awards_gold
0.0    2853
6.0      25
Name: count, dtype: int64

In [ ]:
for coluna in medalhas:
    valores = data[coluna].dropna()
    maximo = (valores == 6).sum()  

    display(f"{coluna}: {maximo} maximo (de {len(valores)} registros)")


'awards_gold: 25 maximo (de 3778 registros)'

'awards_silver: 4 maximo (de 3778 registros)'

'awards_bronze: 11 maximo (de 3778 registros)'

In [ ]:
for coluna in medalhas:
    valores = data[coluna].dropna()
    total_zeros = (valores == 0).sum()  

    display(f"{coluna}: {total_zeros} zeros (de {len(valores)} registros)")

   

'awards_gold: 2853 zeros (de 3778 registros)'

'awards_silver: 2029 zeros (de 3778 registros)'

'awards_bronze: 1274 zeros (de 3778 registros)'

In [244]:
for coluna in medalhas:
    valores = data[coluna].dropna()
    percentual_zeros = valores.isin([0]).mean() * 100

    display(f"{coluna}: {percentual_zeros:.2f}% de zeros")

'awards_gold: 75.52% de zeros'

'awards_silver: 53.71% de zeros'

'awards_bronze: 33.72% de zeros'

## 1.4.2 — Medidas de posição central por pais

As medidas de posicao central do top 10 com pelo menos 10 participacoes nos indicam que, no topo a media e mediana sao proximas em valores, diferindo entre perfis:
- media menor que a mediana = onde parece ter um influencia maior de valores pequenos

- mediana menor que a media = resultados frequentes entre valores medios e alguns excepcionais que aumentam a media.

- algumas diferencas pequenas como a dos EUA podem indicar simetria, a partir de valores adicionais de dispersao. 

In [245]:
pais = data.groupby('country')['awards_gold'].agg(n='count', media='mean', mediana='median')
pais['media_menos_mediana'] = pais['media'] - pais['mediana']

pais = pais[pais['n'] >= 10].sort_values(['mediana', 'media'], ascending=False)
display(pais.head(10))

,n,media,mediana,media_menos_mediana
country,,,,
People's Republic of China,39,4.743590,5.0,-0.256410
Russian Federation,30,3.533333,4.0,-0.466667
United States of America,50,3.020000,3.0,0.020000
Union of Soviet Socialist Republics,29,2.655172,3.0,-0.344828
Republic of Korea,37,2.567568,2.0,0.567568
Democratic People's Republic of Korea,12,1.833333,2.0,-0.166667
Vietnam,48,1.437500,1.0,0.437500
Taiwan,33,1.393939,1.0,0.393939
Hungary,64,1.375000,1.0,0.375000


1.4.3 — Medidas de posição central por anos

In [246]:
resultado = data.groupby('year')[medalhas].agg(['count', 'mean', 'median', 'sum'])

resultado_ordenado = resultado.sort_values(by=(medalhas[0], 'sum'), ascending=False)

display(resultado_ordenado.head(10))

awards_gold                        awards_silver                          \
           count      mean median   sum         count      mean median    sum   
year                                                                            
2011         101  0.534653    0.0  54.0           101  0.891089    0.0   90.0   
2024         108  0.500000    0.0  54.0           108  1.120370    0.5  121.0   
2019         112  0.464286    0.0  52.0           112  0.839286    0.0   94.0   
2021         107  0.485981    0.0  52.0           107  0.962617    0.0  103.0   
2012         100  0.510000    0.0  51.0           100  0.880000    0.0   88.0   
2023         112  0.437500    0.0  49.0           112  0.794643    0.0   89.0   
2009         104  0.471154    0.0  49.0           104  0.942308    0.0   98.0   
2014         101  0.485149    0.0  49.0           101  1.118812    1.0  113.0   
2020         105  0.466667    0.0  49.0           105  1.066667    0.0  112.0   
2017         111  0.432432    0.0  48.0           111  0.810811    0.0   90.0   

     awards_bronze                          
             count      mean median    sum  
year                                        
2011           101  1.356436    1.0  137.0  
2024           108  1.342593    1.0  145.0  
2019           112  1.392857    1.0  156.0  
2021           107  1.383178    1.0  148.0  
2012           100  1.370000    1.0  137.0  
2023           112  1.517857    1.0  170.0  
2009           104  1.298077    1.0  135.0  
2014           101  1.316832    1.0  133.0  
2020           105  1.476190    1.0  155.0  
2017           111  1.378378    1.0  153.0

1.4.4 — Interpretação dos resultados

1.4.5 - resultados do brasil

In [54]:
br = data.query("country == 'Brazil'").copy()
resumo = br[medalhas].agg(["mean", "median"])

moda = br[medalhas].mode().iloc[0]
resumo.loc["mode"] = moda

resumo


,awards_gold,awards_silver,awards_bronze
mean,0.333333,1.288889,2.022222
median,0.000000,1.000000,2.000000
mode,0.000000,0.000000,1.000000


In [59]:
br['decada'] = (br['year'] // 10) * 10
br

resumo_pais = br.groupby('decada')[medalhas].agg(
    ['count','mean','median']
)

ordenado = resumo_pais.sort_index(ascending=False)
ordenado

awards_gold                  awards_silver             awards_bronze  \
             count      mean median         count mean median         count   
decada                                                                        
2020             5  1.000000    1.0             5  2.6    2.0             5   
2010            10  0.200000    0.0            10  2.4    2.5            10   
2000            10  0.200000    0.0            10  1.8    1.5            10   
1990            10  0.300000    0.0            10  0.3    0.0            10   
1980             9  0.333333    0.0             9  0.0    0.0             9   
1970             1  0.000000    0.0             1  0.0    0.0             1   

                         
            mean median  
decada                   
2020    2.000000    2.0  
2010    2.300000    2.5  
2000    3.000000    3.0  
1990    1.400000    1.0  
1980    1.555556    2.0  
1970    0.000000    0.0